In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2
import warnings

# Biogeme core
import biogeme.biogeme as bio
from biogeme import models
from biogeme.expressions import Beta, Variable, bioDraws

import biogeme.database as db

warnings.filterwarnings("ignore")

In [ ]:
amst_data = pd.read_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\amst_processed.csv")

In [ ]:
lnd_data = pd.read_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\lnd_processed.csv")

In [ ]:
mode_to_id = {"car": 1, "pt": 2, "bike": 3}

# MNL Variables
model_columns = [
    "chosen_mode",
    "travel_time_min",
    "distance_km",         
    "n_transfers",           
    "income_quintile",
    "has_driving_license",
    "n_cars_household",
    "age_band",
    "gender",
    "is_peak",
    "weight_trip",
]

In [ ]:
def build_city_workset(city_data, city_name):
    
    ws = city_data[model_columns].copy()

    # 1. Drop rows with any missing value in model columns
    # n_before = len(ws)
    # ws = ws.dropna(subset=model_columns)
    # print(f"{city_name}: dropped {n_before - len(ws):,} rows with missing model vars")

    # 2. Encode dependent variable as integer
    ws["choice_code"] = ws.chosen_mode.map(mode_to_id).astype(int)

    # 3. Cast all numeric columns explicitly
    numeric_cols = [
        "travel_time_min", "distance_km", "n_transfers",
        "income_quintile", "has_driving_license",
        "n_cars_household", "age_band", "gender",
        "is_peak", "weight_trip", "choice_code",
    ]
    ws[numeric_cols] = ws[numeric_cols].apply(pd.to_numeric, errors = "coerce")
    ws = ws.dropna(subset = numeric_cols)

    # 4. Keep only final modelling columns
    keep = ["choice_code"] + [c for c in numeric_cols if c != "choice_code"]
    ws = ws[keep].reset_index(drop=True)

    # 5. Summary
    print(f"{city_name}: {ws.shape[0]:,} rows ready for MNL")
    return ws


# --- Build worksets ---
amsterdam_workset = build_city_workset(amst_data, "Amsterdam")
london_workset    = build_city_workset(lnd_data, "London")

In [ ]:
def estimate_city_mnl_v2(city_workset, city_name, include_peak=False):
    tag = "exp_b_peak" if include_peak else "exp_a_base"
    database = db.Database(f"{city_name.lower()}_{tag}", city_workset)

    # --- Biogeme variable handles ---
    choice = Variable("choice_code")
    travel_time_min = Variable("travel_time_min")
    distance_km = Variable("distance_km")
    income_quintile = Variable("income_quintile")
    # has_driving_lic = Variable("has_driving_license")
    # n_cars = Variable("n_cars_household")
    age_band = Variable("age_band")
    gender = Variable("gender")
    is_peak = Variable("is_peak")

    # ASC_car = 0)
    asc_pt = Beta("asc_pt",   0, None, None, 0)
    asc_bike = Beta("asc_bike", 0, None, None, 0)

    # travel-time coefficient
    b_time_pt = Beta("b_time_pt", 0, None, None, 0)
    b_time_bike = Beta("b_time_bike", 0, None, None, 0)

    # Alternative-specific socioeconomic (car = 0)
    b_income_pt = Beta("b_income_pt",   0, None, None, 0)
    b_income_bike = Beta("b_income_bike", 0, None, None, 0)

    # b_license_pt = Beta("b_license_pt",   0, None, None, 0)
    # b_license_bike = Beta("b_license_bike", 0, None, None, 0)

    # b_cars_pt = Beta("b_cars_pt",   0, None, None, 0)
    # b_cars_bike = Beta("b_cars_bike", 0, None, None, 0)

    b_age_pt = Beta("b_age_pt",   0, None, None, 0)
    b_age_bike = Beta("b_age_bike", 0, None, None, 0)

    b_gender_pt = Beta("b_gender_pt",   0, None, None, 0)
    b_gender_bike = Beta("b_gender_bike", 0, None, None, 0)

    # Mode-specific LoS variables
    b_dist_bike = Beta("b_dist_bike", 0, None, None, 0)   # bike only

    # Optional peak-hour (Experiment B) — alternative-specific ===
    if include_peak:
        b_peak_pt = Beta("b_peak_pt",   0, None, None, 0)
        b_peak_bike = Beta("b_peak_bike", 0, None, None, 0)
    else:
        b_peak_pt = 0
        b_peak_bike = 0

    # Utility functions:
    
    # V_car: reference — only generic time enters
    v_car = 0

    # V_pt: ASC + generic time + socioeconomic (alt-specific) + transfers
    v_pt = (
        asc_pt
        + b_time_pt * travel_time_min
        + b_income_pt * income_quintile
        # + b_license_pt * has_driving_lic
        # + b_cars_pt * n_cars
        + b_age_pt * age_band
        + b_gender_pt * gender
        + b_peak_pt * is_peak
    )

    # V_bike: ASC + generic time + socioeconomic (alt-specific) + distance
    v_bike = (
        asc_bike
        + b_time_bike * travel_time_min
        + b_income_bike * income_quintile
        # + b_license_bike * has_driving_lic
        # + b_cars_bike * n_cars
        + b_age_bike * age_band
        + b_gender_bike * gender
        + b_dist_bike * distance_km
        + b_peak_bike * is_peak
    )

    # --- Choice model ---
    V  = {1: v_car, 2: v_pt, 3: v_bike}
    av = {1: 1, 2: 1, 3: 1}          # all modes available
    logprob = models.loglogit(V, av, choice)

    # --- Biogeme estimation ---
    biogeme_model = bio.BIOGEME(database, logprob, generate_html=False)
    biogeme_model.modelName = f"mnl_{city_name.lower()}_{tag}"
    
    results = biogeme_model.estimate()

    # --- Extract results ---
    beta_table = results.get_estimated_parameters()
    beta_table = beta_table.reset_index().rename(columns = {"index": "parameter"})
    beta_table["city"] = city_name
    beta_table["spec"] = tag
    
    # --- Fit statistics ---
    n_obs = city_workset.shape[0]
    ll_null = -n_obs * np.log(3)

    general_stats = results.get_general_statistics()

    # Extract LL and n_params — values may be tuples like (value, precision_str)
    ll_star = None
    n_params = None
    for key, val in general_stats.items():
        key_str = str(key).lower()
        # val is often a tuple: (numeric_value, format_string)
        numeric_val = val[0] if isinstance(val, (tuple, list)) else val
        if 'final log likelihood' in key_str:
            ll_star = float(numeric_val)
        if 'number of estimated parameters' in key_str:
            n_params = int(numeric_val)

    # Fallback
    if ll_star is None:
        try:
            ll_star = float(results.data.logLike)
        except Exception:
            ll_star = np.nan
    if n_params is None:
        n_params = len(beta_table)

    rho2 = 1.0 - (ll_star / ll_null)

    fit_stats = {
        "city": city_name, "spec": tag,
        "n_obs": n_obs, "n_params": n_params,
        "LL_null": round(ll_null, 2),
        "LL_star": round(ll_star, 2),
        "rho2": round(rho2, 4),
    }

    # Print Resutlts
    print(f"\n{'='*60}")
    print(f"✓ {city_name} | {tag} | n={n_obs:,} | LL*={ll_star:.2f} | ρ²={rho2:.4f}")
    print(f"{'='*60}")
    print(beta_table.to_string(index=False))

    return biogeme_model, results, beta_table, fit_stats

In [ ]:
amst_model_a, amst_results_a, amst_beta_a, amst_fit_a = estimate_city_mnl_v2(
    amsterdam_workset, "Amsterdam", include_peak = False
)

In [ ]:
amst_model_b, amst_results_b, amst_beta_b, amst_fit_b = estimate_city_mnl_v2(
    amsterdam_workset, "Amsterdam", include_peak = True
)

In [ ]:
lnd_model_a, lnd_results_a, lnd_beta_a, lnd_fit_a = estimate_city_mnl_v2(
    london_workset, "London", include_peak = False
)

In [ ]:
lnd_model_b, lnd_results_b, lnd_beta_b, lnd_fit_b = estimate_city_mnl_v2(
    london_workset, "London", include_peak = True
)

In [ ]:
def lr_test(fit_a, fit_b, alpha=0.05):
    """
    Likelihood-ratio test for nested models.
    H0: restricted model (A) is sufficient.
    H1: unrestricted model (B) fits significantly better.
    """
    lr_stat = -2 * (fit_a["LL_star"] - fit_b["LL_star"])
    delta_k = fit_b["n_params"] - fit_a["n_params"]
    p_value = chi2.sf(lr_stat, df=delta_k)
    reject  = p_value < alpha

    print(f" LR statistic : {lr_stat:.4f}")
    print(f" delta_k (df) : {delta_k}")
    print(f" p-value : {p_value:.6f}")
    print(f" Critical chi^2 : {chi2.ppf(1 - alpha, delta_k):.3f}")
    print(f" Decision : {'Reject H0 → select Exp B' if reject else 'Fail to reject H0, so we select Exp A'}")
    return reject, lr_stat, p_value


print("\nAMSTERDAM: LR Test (Exp A vs Exp B)")
amst_select_b, _, _ = lr_test(amst_fit_a, amst_fit_b)

print("\nLONDON: LR Test (Exp A vs Exp B)")
lnd_select_b, _, _ = lr_test(lnd_fit_a, lnd_fit_b)

In [ ]:
# --- Store final selections ---
amst_final_tag   = "exp_b_peak" if amst_select_b else "exp_a_base"
amst_final_beta  = amst_beta_b  if amst_select_b else amst_beta_a
amst_final_fit   = amst_fit_b   if amst_select_b else amst_fit_a
amst_final_res   = amst_results_b if amst_select_b else amst_results_a

lnd_final_tag    = "exp_b_peak" if lnd_select_b else "exp_a_base"
lnd_final_beta   = lnd_beta_b   if lnd_select_b else lnd_beta_a
lnd_final_fit    = lnd_fit_b    if lnd_select_b else lnd_fit_a
lnd_final_res    = lnd_results_b if lnd_select_b else lnd_results_a

print(f"\n✓ Amsterdam final model: {amst_final_tag}")
print(f"✓ London final model:    {lnd_final_tag}")

In [ ]:
def format_coeff_table(beta_df, city_label):
    """Format one city's beta table for merging."""
    cols = ["Name", "Value", "Robust std err.", "Robust t-stat.", "Robust p-value"]
    out = beta_df[cols].copy()
    out = out.rename(columns={
        "Value":         f"{city_label}_coeff",
        "Robust std err.":  f"{city_label}_se",
        "Robust t-stat.":   f"{city_label}_t",
        "Robust p-value":  f"{city_label}_p",
    })
    # Round for readability
    for c in out.columns[1:]:
        out[c] = out[c].round(4)
    return out


amst_fmt = format_coeff_table(amst_final_beta, "AMS")
lnd_fmt  = format_coeff_table(lnd_final_beta, "LDN")

In [ ]:
# Merge on parameter name (outer join to keep all params from both)
comparison = amst_fmt.merge(lnd_fmt, on = "Name", how = "outer").set_index("Name")

# Reorder columns for readability: AMS coeff, SE, t, p | LDN coeff, SE, t, p
col_order = [
    "AMS_coeff", "AMS_se", "AMS_t", "AMS_p",
    "LDN_coeff", "LDN_se", "LDN_t", "LDN_p",
]
comparison = comparison[[c for c in col_order if c in comparison.columns]]

print("\n SIDE-BY-SIDE COEFFICIENT TABLE (Final Models)\n")
print(comparison.to_string())

In [ ]:
# --- Goodness-of-fit summary ---
fit_summary = pd.DataFrame([amst_final_fit, lnd_final_fit])
fit_summary = fit_summary.set_index("city")[["spec", "n_obs", "n_params", "LL_null", "LL_star", "rho2"]]

print("\n GOODNESS-OF-FIT SUMMARY\n ")
print(fit_summary.to_string())

In [ ]:
def extract_param(beta_df, param_name):
    """Extract estimate and SE for a single parameter."""
    row = beta_df.loc[beta_df["Name"] == param_name]
    if row.empty:
        return np.nan, np.nan
    return row["Value"].values[0], row["Robust std err."].values[0]
    
def analyze_hypothesis(param_name, ams_beta, lnd_beta, title):
    print(f"\n{title}")
    
    b_ams, se_ams = extract_param(ams_beta, param_name)
    b_lnd, se_lnd = extract_param(lnd_beta, param_name)

    # Confidence Intervals Calculation
    ci_ams = (b_ams - 1.96 * se_ams, b_ams + 1.96 * se_ams)
    ci_lnd = (b_lnd - 1.96 * se_lnd, b_lnd + 1.96 * se_lnd)

    # 3. Results
    print(f"  Amsterdam  {param_name} = {b_ams:.4f} (SE={se_ams:.4f}) 95% CI: [{ci_ams[0]:.4f}, {ci_ams[1]:.4f}]")
    print(f"  London     {param_name} = {b_lnd:.4f} (SE={se_lnd:.4f}) 95% CI: [{ci_lnd[0]:.4f}, {ci_lnd[1]:.4f}]")

    # 4. Overlap Logic
    overlap = ci_ams[1] >= ci_lnd[0] and ci_lnd[1] >= ci_ams[0]
    print(f"  CIs overlap: {overlap}")
    
    if not overlap:
        print(" Coefficients are significantly different at 95% level.")
    else:
        print(" CIs overlap - formal Wald test needed")

In [ ]:
# H1 Analysis
analyze_hypothesis("b_time_pt", amst_final_beta, lnd_final_beta, "H1 — TRAVEL TIME SENSITIVITY (PT)")
analyze_hypothesis("b_time_bike", amst_final_beta, lnd_final_beta, "H1 — TRAVEL TIME SENSITIVITY (BIKE)")

# H2 Analysis
print(f"\nH2 — AGE & INCOME EFFECT\n")
h2_params = ["b_age_pt", "b_age_bike", "b_income_pt", "b_income_bike"]

for param in h2_params:
    val_a, se_a = extract_param(amst_final_beta, param)
    val_l, se_l = extract_param(lnd_final_beta, param)
    print(f"  {param:15s} | AMS: {val_a:+.4f} (SE={se_a:.4f}) | LDN: {val_l:+.4f} (SE={se_l:.4f})")

In [ ]:
# --- Export ---
amst_final_beta.to_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\amst_final_beta.csv", index=False)
print(f"\n Amsterdam model saved to data\processed")

In [ ]:
# --- Export ---
lnd_final_beta.to_csv(r"C:\Users\vladimir.koev\OneDrive - Трансметрикс АД\SoftUni\Data Science\Final Exam\data\processed\lnd_final_beta.csv", index=False)
print(f"\n London model saved to data\processed")